# Dimensional Model Design

## Selected Business Processes

The main business processes selected from the OLTP system are:

1. Film Rental Transactions
2. Payment Transactions

These processes are important because they capture the main operational activities of the movie rental business and provide measurable analytical insights.

---

## Fact Tables

### FactRental

**Business Process:** Film rental transactions

**Grain:** One row per rental transaction — one customer rents one film at one point in time.

**Measures:**
- rental_duration_days (actual days the film was kept)
- is_late (1 = late return, 0 = on time)
- late_days (number of days overdue)

**Foreign Keys:**
- date_key
- customer_key
- film_key
- category_key
- store_key
- staff_key

---

### FactPayment

**Business Process:** Payment transactions

**Grain:** One row per payment transaction — one customer makes one payment for one rental.

**Measures:**
- amount (payment amount in USD)

**Foreign Keys:**
- date_key
- customer_key
- store_key
- staff_key
- rental_id

---

## Dimension Tables

### DimDate
Stores date-related attributes for trend analysis.

Attributes:
- date_key
- full_date
- day
- month
- quarter
- year
- day_of_week
- is_weekend

Source:
Generated from transaction dates in rental and payment datasets.

---

### DimCustomer

Attributes:
- customer_key
- customer_id
- full_name
- email
- address
- city
- country
- store_id
- active

Source Tables:
- customer
- address
- city
- country

---

### DimFilm

Attributes:
- film_key
- film_id
- title
- release_year
- rental_rate
- rating
- length
- language

Source Tables:
- film
- language

---

### DimCategory

Attributes:
- category_key
- category_name

Source Tables:
- category
- film_category

Note: Categories are linked to fact_rental through films 
using the film_category mapping table. Film-category 
relationships were resolved during transformation 
using the film_category bridge table.

---

### DimStore

Attributes:
- store_key
- store_id
- address
- city
- country

Source Tables:
- store
- address
- city
- country

---

### DimStaff

Attributes:
- staff_key
- staff_id
- full_name
- store_id

Source Table:
- staff

---

### DimLocation

DimLocation centralizes geographical attributes from 
address, city, and country tables for location-based analysis.

It supports analytical questions such as:
- Which cities generate the highest rental activity?
- Which countries have the most active customers?

Note: DimLocation is prepared as a supporting dimension
for future analytical expansion. Location attributes
are currently embedded in DimCustomer and DimStore 
for direct analysis.

---

### DimLanguage

Language attribute is integrated within DimFilm.
Although a language table exists in the OLTP system,
all 1,000 films share the same language (English),
therefore language was embedded in DimFilm as an
attribute rather than a separate dimension.

---

### DimActor

DimActor stores actor information extracted from the actor table.

Attributes:
- actor_key
- actor_id
- full_name

Source Table:
- actor

Note: Since films and actors have a many-to-many relationship
through the film_actor bridge table, DimActor was prepared
for future analytical expansion. Full integration would require
a bridge table linking DimActor to fact_rental through film_actor.

---

## Schema Type

A Star Schema is selected because it simplifies analytical queries 
and improves reporting performance by organizing data around 
central fact tables connected to descriptive dimensions.

In [17]:
import pymysql
import pandas as pd

conn = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="root123",
    database="sakila",
    connect_timeout=5
)
cursor = conn.cursor()
print("Connected ✅")

Connected ✅


In [21]:
# dim_date
from datetime import datetime, timedelta

start_date = datetime(2005, 1, 1)
end_date = datetime(2006, 12, 31)
dates = []
current = start_date

while current <= end_date:
    dates.append({
        'date_key': int(current.strftime('%Y%m%d')),
        'full_date': current.date(),
        'day': current.day,
        'month': current.month,
        'month_name': current.strftime('%B'),
        'quarter': (current.month - 1) // 3 + 1,
        'year': current.year,
        'day_of_week': current.strftime('%A'),
        'is_weekend': 1 if current.weekday() >= 5 else 0
    })
    current += timedelta(days=1)

dim_date = pd.DataFrame(dates)
print(f"dim_date Ready! Rows: {len(dim_date)}")
print(dim_date.head(3))

dim_date Ready! Rows: 730
   date_key   full_date  day  month month_name  quarter  year day_of_week  \
0  20050101  2005-01-01    1      1    January        1  2005    Saturday   
1  20050102  2005-01-02    2      1    January        1  2005      Sunday   
2  20050103  2005-01-03    3      1    January        1  2005      Monday   

   is_weekend  
0           1  
1           1  
2           0  


In [ ]:
# dim_customer
cursor.execute("""
    SELECT 
        c.customer_id,
        CONCAT(c.first_name, ' ', c.last_name) as full_name,
        c.email,
        a.address,
        ci.city,
        co.country,
        c.store_id,
        c.active
    FROM customer c
    JOIN address a ON c.address_id = a.address_id
    JOIN city ci ON a.city_id = ci.city_id
    JOIN country co ON ci.country_id = co.country_id
""")
rows = cursor.fetchall()
dim_customer = pd.DataFrame(rows, columns=[
    'customer_id', 'full_name', 'email',
    'address', 'city', 'country', 'store_id', 'active'])
dim_customer.insert(0, 'customer_key', range(1, len(dim_customer) + 1))
print(f"dim_customer Ready! Rows: {len(dim_customer)}")

# dim_film
cursor.execute("""
    SELECT 
        f.film_id, f.title, f.description,
        f.release_year, l.name as language,
        f.rental_duration, f.rental_rate,
        f.rating, f.length
    FROM film f
    JOIN language l ON f.language_id = l.language_id
""")
rows = cursor.fetchall()
dim_film = pd.DataFrame(rows, columns=[
    'film_id', 'title', 'description', 'release_year',
    'language', 'rental_duration', 'rental_rate', 'rating', 'length'])
dim_film.insert(0, 'film_key', range(1, len(dim_film) + 1))
print(f"dim_film Ready! Rows: {len(dim_film)}")

# dim_category
cursor.execute("SELECT category_id, name FROM category")
rows = cursor.fetchall()
dim_category = pd.DataFrame(rows, columns=['category_id', 'category_name'])
dim_category.insert(0, 'category_key', range(1, len(dim_category) + 1))
print(f"dim_category Ready! Rows: {len(dim_category)}")

# dim_store
cursor.execute("""
    SELECT s.store_id, a.address, ci.city, co.country
    FROM store s
    JOIN address a ON s.address_id = a.address_id
    JOIN city ci ON a.city_id = ci.city_id
    JOIN country co ON ci.country_id = co.country_id
""")
rows = cursor.fetchall()
dim_store = pd.DataFrame(rows, columns=['store_id', 'address', 'city', 'country'])
dim_store.insert(0, 'store_key', range(1, len(dim_store) + 1))
print(f"dim_store Ready! Rows: {len(dim_store)}")

# dim_staff
cursor.execute("""
    SELECT s.staff_id,
           CONCAT(s.first_name, ' ', s.last_name) as full_name,
           s.email, s.store_id
    FROM staff s
""")
rows = cursor.fetchall()
dim_staff = pd.DataFrame(rows, columns=['staff_id', 'full_name', 'email', 'store_id'])
dim_staff.insert(0, 'staff_key', range(1, len(dim_staff) + 1))
print(f"dim_staff Ready! Rows: {len(dim_staff)}")

_IncompleteInputError: incomplete input (3015283298.py, line 70)

In [18]:
# dim_actor
cursor.execute("""
    SELECT 
        a.actor_id,
        CONCAT(a.first_name, ' ', a.last_name) as full_name
    FROM actor a
""")
rows = cursor.fetchall()
dim_actor = pd.DataFrame(rows, columns=['actor_id', 'full_name'])
dim_actor.insert(0, 'actor_key', range(1, len(dim_actor) + 1))

print("dim_actor READY!")
print(f" Rows: {len(dim_actor)}")
print(dim_actor.head(3))

# dim_location
cursor.execute("""
    SELECT DISTINCT
        ci.city_id,
        ci.city,
        co.country
    FROM city ci
    JOIN country co ON ci.country_id = co.country_id
""")
rows = cursor.fetchall()
dim_location = pd.DataFrame(rows, columns=['city_id', 'city', 'country'])
dim_location.insert(0, 'location_key', range(1, len(dim_location) + 1))

print("\ndim_location Ready!")
print(f"Rows: {len(dim_location)}")
print(dim_location.head(3))

dim_actor READY!
 Rows: 200
   actor_key  actor_id         full_name
0          1         1  PENELOPE GUINESS
1          2         2     NICK WAHLBERG
2          3         3          ED CHASE

dim_location Ready!
Rows: 600
   location_key  city_id    city      country
0             1      251   Kabul  Afghanistan
1             2       59   Batna      Algeria
2             3       63  Béchar      Algeria


In [20]:
import os
path = r'C:\Users\User\OneDrive\Desktop\movie_rental_dw\data\\'
dim_actor.to_csv(path + 'dim_actor.csv', index=False)
dim_location.to_csv(path + 'dim_location.csv', index=False)
print("Saved ✅")

Saved ✅
